# MyUPI Voice Assistant on Google Colab (GPU)

This notebook sets up and runs the MyUPI Voice Assistant.
- **TTS removed**: Responses returned cleanly in chat interface.
- **Persistent model cache**: Model downloads once to /content/asr_model and is reused.
- **Zero double-loading**: Server loads the model directly from disk into GPU without redundant memory overhead.


### Step 1: Install Dependencies (Run once per Colab session)

In [ ]:
!apt-get install -y ffmpeg > /dev/null
!pip install -q Cython nemo_toolkit[asr]
!pip install -q fastapi==0.115.12 uvicorn[standard]==0.32.1 sqlalchemy[asyncio] aiosqlite pydantic-settings rapidfuzz indic-transliteration
!pip install -q protobuf==7.36.2
print("Dependencies installed!")

### Step 2: Download Model to Disk (Run once per Colab session)
*This only downloads model files to disk. It does NOT load into Python memory, preserving RAM.*

In [ ]:
import os, getpass
from huggingface_hub import snapshot_download

if not os.environ.get("HF_TOKEN"):
    print("Paste your HuggingFace Token:")
    os.environ["HF_TOKEN"] = getpass.getpass()

model_dir = "/content/asr_model"
nemo_file = os.path.join(model_dir, "nemo", "indic_transcribe_flex.nemo")

if os.path.exists(nemo_file):
    print(f"ASR Model files ALREADY downloaded at {model_dir}!")
else:
    print(f"Downloading ASR Model files to {model_dir} (one time only)...")
    snapshot_download(
        repo_id="bodhan-ai/indic-transcribe-flex",
        local_dir=model_dir,
        token=os.environ["HF_TOKEN"],
    )
    print(f"Model files successfully saved to {model_dir}!")


### Step 3: Clone/Pull Code & Launch Server
*Run this cell whenever you want to start the app or test code updates.*

In [ ]:
import os, subprocess, time
os.environ["ASR_MODEL_DIR"] = "/content/asr_model"

# 1. Clone or pull repository
if not os.path.exists("/content/myupi-voice-action"):
    !git clone https://github.com/ansh63766/myupi-voice-action.git /content/myupi-voice-action
    %cd /content/myupi-voice-action
else:
    %cd /content/myupi-voice-action
    !git pull origin master

# 2. Kill any ghost process on port 8000
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)

# 3. Create Google proxy link
from google.colab import output
print("\n" + "="*60)
print("CLICK THIS SECURE GOOGLE LINK TO OPEN APP")
output.serve_kernel_port_as_window(8000)
print("="*60 + "\n")

# 4. Start Server (loads model ONCE from /content/asr_model in ~25s)
!uvicorn app.main:app --host 0.0.0.0 --port 8000
